In [1]:
import numpy as np
import open3d as o3d
from pathlib import Path
import torch
import pickle
import os


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
UHM_DATASET_DIR = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train")


In [3]:
print(os.path.exists(UHM_DATASET_DIR))
print(os.path.isdir(UHM_DATASET_DIR))

True
True


In [4]:
DATASET_DIR = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/faces")
os.makedirs(DATASET_DIR, exist_ok=True)

In [5]:
print(os.path.exists(DATASET_DIR))
print(os.path.isdir(DATASET_DIR))

True
True


In [6]:
all_files = list(UHM_DATASET_DIR.glob("*.ply"))

In [7]:
pca_data = torch.load("pca_basis_all.pth")
all_gt_z = pca_data['gt_z']
all_sorted_filenames = sorted([f.name for f in UHM_DATASET_DIR.glob("*.ply")])
name_to_idx = {name: idx for idx, name in enumerate(all_sorted_filenames)}
print(f"Loaded GT Z tensor of shape: {all_gt_z.shape}")

Loaded GT Z tensor of shape: torch.Size([32, 8000, 100])


In [8]:
RANDOM_SEED = 42

In [9]:
np.random.seed(RANDOM_SEED)

In [10]:
N_SAMPLES = 2500  # 80/10/10 split → 2000 train, 250 val, 250 test

In [11]:
random_sample = np.random.choice(all_files, size=N_SAMPLES, replace=False)

In [12]:
random_sample

array([WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/3478.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/3890.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/2871.ply'),
       ...,
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/784.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/6136.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/8990.ply')],
      dtype=object)

In [13]:
rng = np.random.default_rng(RANDOM_SEED)

In [14]:
"""
def generate_random_view(
    pcd_points: np.ndarray, plane_width=0.001, plane_height_factor=1.2, plane_depth_factor=1.2, downsample_p=0.5, side_tol=1e-9
) -> np.ndarray:

    center = pcd_points.mean(axis=0)
    extent_y = np.ptp(pts[:, 1])
    extent_z = np.ptp(pts[:, 2])
    height = extent_y * plane_height_factor
    depth = extent_z * plane_depth_factor

    plane = o3d.geometry.TriangleMesh.create_box(width=plane_width, height=height, depth=depth)
    plane.translate([center[0] - plane_width / 2, center[1] - height / 2, center[2] - depth / 2])

    x_angle = rng.uniform(0, 2 * np.pi)
    y_angle = rng.uniform(0, 2 * np.pi)
    z_angle = rng.uniform(0, 2 * np.pi)
    R_plane = o3d.geometry.get_rotation_matrix_from_xyz([x_angle, y_angle, z_angle])
    plane.rotate(R_plane, center=center)

    initial_normal = np.array([1.0, 0.0, 0.0])
    rotated_normal = R_plane @ initial_normal
    rotated_normal /= np.linalg.norm(rotated_normal)

    vecs = pcd_points - center[np.newaxis, :]
    signed = vecs.dot(rotated_normal)
    side = rng.choice([1, -1])
    mask_side = (signed * side) > side_tol
    selected_pts = pcd_points[mask_side]
    orig_indices = np.nonzero(mask_side)[0]

    down_mask_bool = rng.choice([False, True], size=selected_pts.shape[0],
                                p=[1 - downsample_p, downsample_p])
    downsampled_pts = selected_pts[down_mask_bool]
    kept_indices = orig_indices[down_mask_bool]

    angles = rng.uniform(0, 2 * np.pi, size=3)
    R = o3d.geometry.get_rotation_matrix_from_xyz(angles)
    t = rng.uniform(-1.0, 1.0, size=3)
    transformed_points = (R @ downsampled_pts.T).T + t

    T = np.eye(4, dtype=float)
    T[:3, :3] = R
    T[:3, 3] = t

    R_inv = R.T
    t_inv = -R_inv @ t
    T_inv = np.eye(4, dtype=float)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv

    print(f"Reproyection error T_inv {np.linalg.norm((np.hstack((transformed_points, np.ones((transformed_points.shape[0], 1)))) @ T_inv.T)[:, :3] - pts[kept_indices])}")

    return transformed_points, R_inv, t_inv, plane, kept_indices
"""

'\ndef generate_random_view(\n    pcd_points: np.ndarray, plane_width=0.001, plane_height_factor=1.2, plane_depth_factor=1.2, downsample_p=0.5, side_tol=1e-9\n) -> np.ndarray:\n\n    center = pcd_points.mean(axis=0)\n    extent_y = np.ptp(pts[:, 1])\n    extent_z = np.ptp(pts[:, 2])\n    height = extent_y * plane_height_factor\n    depth = extent_z * plane_depth_factor\n\n    plane = o3d.geometry.TriangleMesh.create_box(width=plane_width, height=height, depth=depth)\n    plane.translate([center[0] - plane_width / 2, center[1] - height / 2, center[2] - depth / 2])\n\n    x_angle = rng.uniform(0, 2 * np.pi)\n    y_angle = rng.uniform(0, 2 * np.pi)\n    z_angle = rng.uniform(0, 2 * np.pi)\n    R_plane = o3d.geometry.get_rotation_matrix_from_xyz([x_angle, y_angle, z_angle])\n    plane.rotate(R_plane, center=center)\n\n    initial_normal = np.array([1.0, 0.0, 0.0])\n    rotated_normal = R_plane @ initial_normal\n    rotated_normal /= np.linalg.norm(rotated_normal)\n\n    vecs = pcd_points -

In [15]:
import numpy as np

def generate_random_view(
    pcd_points: np.ndarray,
    index: int,
    downsample_p=0.5,
    **kwargs
):
    """Randomized face slice with uniformly random SO(3) rotation and random translation.
    GT transform (R_inv, t_inv) maps the saved transformed points back to face-forward frame.
    """
    rng = np.random.default_rng()

    X_MIN_BASE, X_MAX_BASE = -0.85,  0.85
    Y_MIN_BASE, Y_MAX_BASE = -0.5,   1.1
    Z_MIN_BASE             = -0.2

    X_MIN = X_MIN_BASE * rng.uniform(0.65, 1.35)
    X_MAX = X_MAX_BASE * rng.uniform(0.65, 1.35)
    Y_MIN = Y_MIN_BASE * rng.uniform(0.65, 1.35)
    Y_MAX = Y_MAX_BASE * rng.uniform(0.65, 1.35)
    Z_MIN = Z_MIN_BASE * rng.uniform(0.65, 1.35)

    mask = (
        (pcd_points[:, 0] >= X_MIN) & (pcd_points[:, 0] <= X_MAX) &
        (pcd_points[:, 1] >= Y_MIN) & (pcd_points[:, 1] <= Y_MAX) &
        (pcd_points[:, 2] >= Z_MIN)
    )

    orig_indices = np.where(mask)[0]
    selected_pts = pcd_points[orig_indices]

    if len(selected_pts) == 0:
        orig_indices = np.arange(len(pcd_points))
        selected_pts = pcd_points

    if downsample_p < 1.0:
        keep = rng.choice([False, True], size=len(selected_pts),
                          p=[1 - downsample_p, downsample_p])
        selected_pts = selected_pts[keep]
        orig_indices = orig_indices[keep]

    # Uniformly random SO(3) rotation via QR decomposition of a random Gaussian matrix
    A = rng.standard_normal((3, 3))
    Q, _ = np.linalg.qr(A)
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1
    R = Q  # uniform SO(3)

    t = rng.uniform(-1.0, 1.0, size=3)

    transformed = (R @ selected_pts.T).T + t

    R_inv = R.T
    t_inv = -R.T @ t

    print(f"index={index}  X=[{X_MIN:.2f},{X_MAX:.2f}] Y=[{Y_MIN:.2f},{Y_MAX:.2f}] "
          f"Z>={Z_MIN:.2f}  pts={len(orig_indices)}")

    return transformed.astype(np.float32), R_inv, t_inv, None, orig_indices


In [16]:
random_file = random_sample[0]

In [17]:
pcd = o3d.io.read_point_cloud(str(random_file))
pcd.colors = o3d.utility.Vector3dVector(np.ones((len(pcd.points), 3)) * 0.5)
pts = np.asarray(pcd.points)

In [18]:
#view_points, view_r, view_t, view_plane, kept_indices = generate_random_view(pts)

view_points, view_r, view_t, view_camera, kept_indices = generate_random_view(pts, 1)

index=1  X=[-1.12,0.62] Y=[-0.50,1.14] Z>=-0.14  pts=3401


In [19]:
view_r.dtype

dtype('float64')

In [20]:
view_pcd = o3d.geometry.PointCloud()
view_pcd.points = o3d.utility.Vector3dVector(view_points)

In [21]:
#o3d.visualization.draw_plotly([view_pcd, view_plane, pcd])

o3d.visualization.draw_plotly([view_pcd, pcd])

In [22]:
T = np.eye(4, dtype=np.float64)
T[:3, :3] = view_r
T[:3, 3] = view_t

In [23]:
transformed_view = view_pcd.transform(T)

In [24]:
#o3d.visualization.draw_plotly([transformed_view, view_plane])

o3d.visualization.draw_plotly([transformed_view])

In [25]:
def build_metadata_dict(scene_name, pcd0_path, pcd1_path, pcd_morphed_path, gt_z_path, R, t, frag_id1, kept_indices):
    metadata = {
        "overlap": 0,
        "pcd0": str(pcd0_path),
        "pcd1": str(pcd1_path),
        "pcd_morphed": str(pcd_morphed_path),
        "gt_z_path": str(gt_z_path),
        "rotation": R,
        "translation": t,
        "scene_name": scene_name,
        "frag_id0": 0,
        "frag_id1": frag_id1,
        "kept_indices": kept_indices,
    }
    return metadata


In [26]:
FULL_PC_NAME =  "full_face.pth"
MORPHED_PC_NAME = "full_morphed_face.pth"
AVERAGE_FACE_REL = "average_face.pth"   # single shared ref, relative to data_root

print(f"Computing average face from {len(all_files)} files...")
all_pts_list = []

for f in all_files:
    temp_pcd = o3d.io.read_point_cloud(str(f))
    all_pts_list.append(np.asarray(temp_pcd.points))

average_pts = np.mean(np.stack(all_pts_list), axis=0)
print(f"Average point cloud created. Shape: {average_pts.shape}")

# Save once — every sample's pcd0 will point here
avg_face_path = DATASET_DIR / "data" / AVERAGE_FACE_REL
avg_face_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(average_pts.astype(np.float32), avg_face_path)
print(f"Saved average face → {avg_face_path}")

Computing average face from 8000 files...
Average point cloud created. Shape: (10788, 3)
Saved average face → C:\Eli Folder temp\geotransformer-faces-updated\data\faces\data\average_face.pth


In [27]:
def process_file(file_path: Path, n_views=2, folder="train", save=True):
    pcd = o3d.io.read_point_cloud(str(file_path))
    pts = np.asarray(pcd.points)

    print(f"Processing file: {file_path.stem} with {pts.shape[0]} points.")

    subject_path = DATASET_DIR / "data" / folder / file_path.stem

    file_idx = name_to_idx[file_path.name]
    # .clone() is critical: all_gt_z[:, file_idx, :] is a view into [32, 8000, 100].
    # Without clone(), torch.save serializes the full 102 MB backing storage instead of
    # the 13 KB slice.
    sample_gt_z = all_gt_z[:, file_idx, :].clone()

    if save:
        subject_path.mkdir(parents=True, exist_ok=True)
        torch.save(pts.astype(np.float32), subject_path / MORPHED_PC_NAME)
        torch.save(sample_gt_z, subject_path / "gt_z.pth")

    metadata_list = []

    for i in range(n_views):
        transformed_points, R_inv, t_inv, _, kept_indices = generate_random_view(pts, i)

        metadata = build_metadata_dict(
            scene_name=file_path.stem,
            pcd0_path=AVERAGE_FACE_REL,
            pcd1_path=(Path(folder) / file_path.stem / f"view_{i+1}.pth").as_posix(),
            pcd_morphed_path=(Path(folder) / file_path.stem / MORPHED_PC_NAME).as_posix(),
            gt_z_path=(Path(folder) / file_path.stem / "gt_z.pth").as_posix(),
            R=R_inv,
            t=t_inv,
            frag_id1=i + 1,
            kept_indices=kept_indices
        )
        metadata_list.append(metadata)
        if save:
            torch.save(transformed_points, subject_path / f"view_{i+1}.pth")

    return metadata_list


In [28]:
train_size = int(np.round(0.8 * len(random_sample)))
val_size = int(np.round(0.1 * len(random_sample)))
test_size = len(random_sample) - train_size - val_size

In [29]:
train_size, val_size, test_size

(2000, 250, 250)

In [30]:
train_metadata = []
for file_path in random_sample[:train_size]:
    metadata_list = process_file(file_path, n_views=2, folder="train")
    train_metadata.extend(metadata_list)
val_metadata = []
for file_path in random_sample[train_size:train_size+val_size]:
    metadata_list = process_file(file_path, n_views=2, folder="val")
    val_metadata.extend(metadata_list)

Processing file: 3478 with 10788 points.
index=0  X=[-0.85,0.85] Y=[-0.42,0.97] Z>=-0.24  pts=3418
index=1  X=[-0.91,0.99] Y=[-0.45,1.08] Z>=-0.17  pts=3338
Processing file: 3890 with 10788 points.
index=0  X=[-0.58,0.76] Y=[-0.39,0.95] Z>=-0.21  pts=3323
index=1  X=[-0.97,0.74] Y=[-0.51,1.28] Z>=-0.16  pts=3451
Processing file: 2871 with 10788 points.
index=0  X=[-0.98,0.66] Y=[-0.46,1.22] Z>=-0.25  pts=3479
index=1  X=[-0.82,1.02] Y=[-0.50,1.42] Z>=-0.25  pts=3592
Processing file: 440 with 10788 points.
index=0  X=[-1.00,0.67] Y=[-0.52,0.73] Z>=-0.25  pts=3190
index=1  X=[-0.71,0.73] Y=[-0.62,1.18] Z>=-0.22  pts=3591
Processing file: 5867 with 10788 points.
index=0  X=[-0.91,0.72] Y=[-0.52,1.23] Z>=-0.22  pts=3640
index=1  X=[-0.67,1.02] Y=[-0.58,1.02] Z>=-0.22  pts=3607
Processing file: 4 with 10788 points.
index=0  X=[-0.92,0.74] Y=[-0.34,0.89] Z>=-0.15  pts=3030
index=1  X=[-0.57,0.73] Y=[-0.43,1.22] Z>=-0.21  pts=3450
Processing file: 2987 with 10788 points.
index=0  X=[-1.15,0.9

In [31]:
test_metadata = []
for file_path in random_sample[train_size+val_size:]:
    metadata_list = process_file(file_path, n_views=1, folder="test")
    test_metadata.extend(metadata_list)

Processing file: 108 with 10788 points.
index=0  X=[-0.90,0.93] Y=[-0.35,1.08] Z>=-0.22  pts=3262
Processing file: 8828 with 10788 points.
index=0  X=[-0.73,0.92] Y=[-0.59,1.31] Z>=-0.17  pts=3519
Processing file: 1472 with 10788 points.
index=0  X=[-0.59,0.84] Y=[-0.47,1.37] Z>=-0.16  pts=3432
Processing file: 5571 with 10788 points.
index=0  X=[-0.87,0.93] Y=[-0.54,1.44] Z>=-0.23  pts=3682
Processing file: 5964 with 10788 points.
index=0  X=[-0.59,0.57] Y=[-0.55,1.05] Z>=-0.15  pts=3416
Processing file: 9787 with 10788 points.
index=0  X=[-0.97,0.75] Y=[-0.43,0.94] Z>=-0.19  pts=3226
Processing file: 5155 with 10788 points.
index=0  X=[-1.01,1.02] Y=[-0.49,1.39] Z>=-0.22  pts=3501
Processing file: 7412 with 10788 points.
index=0  X=[-0.92,0.60] Y=[-0.34,1.21] Z>=-0.13  pts=3210
Processing file: 6980 with 10788 points.
index=0  X=[-1.04,0.72] Y=[-0.66,1.19] Z>=-0.14  pts=3541
Processing file: 842 with 10788 points.
index=0  X=[-0.71,0.74] Y=[-0.61,1.30] Z>=-0.26  pts=3809
Processing f

In [32]:
METADATA_DIR = DATASET_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

In [33]:
with open(METADATA_DIR / "train.pkl", "wb") as f:
    pickle.dump(train_metadata, f)

In [34]:
with open(METADATA_DIR / "val.pkl", "wb") as f:
    pickle.dump(val_metadata, f)

In [35]:
demo_folder = DATASET_DIR / "demo"
demo_folder.mkdir(parents=True, exist_ok=True)

In [36]:
for ex_id, example in enumerate(test_metadata):
    example_ref = torch.load(DATASET_DIR / "data" / example["pcd0"], weights_only=False)
    example_src = torch.load(DATASET_DIR / "data" / example["pcd1"], weights_only=False)
    example_gt_morphed = torch.load(DATASET_DIR / "data" / example["pcd_morphed"], weights_only=False)
    np.save(demo_folder / f"ref_{ex_id}.npy", example_ref)
    np.save(demo_folder / f"src_{ex_id}.npy", example_src)
    np.save(demo_folder / f"morphed_full_{ex_id}.npy", example_gt_morphed)
    rot = example["rotation"]
    t = example["translation"]
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = rot
    T[:3, 3] = t
    np.save(demo_folder / f"gt_{ex_id}.npy", T)


### Inspect Test Data

In [37]:
#test_sample = test_metadata[np.random.randint(len(test_metadata))]
test_sample = test_metadata[0]

In [38]:
test_sample

{'overlap': 0,
 'pcd0': 'average_face.pth',
 'pcd1': 'test/108/view_1.pth',
 'pcd_morphed': 'test/108/full_morphed_face.pth',
 'gt_z_path': 'test/108/gt_z.pth',
 'rotation': array([[-0.76259712,  0.23206374, -0.60381459],
        [ 0.20252667, -0.80085295, -0.56357564],
        [-0.61435217, -0.55206972,  0.56372905]]),
 'translation': array([ 0.21748095,  0.13045849, -0.71944305]),
 'scene_name': '108',
 'frag_id0': 0,
 'frag_id1': 1,
 'kept_indices': array([    0,    18,    21, ..., 10763, 10766, 10767])}

In [39]:
test_src = torch.load(DATASET_DIR / "data" / test_sample["pcd1"], weights_only=False)
test_ref = torch.load(DATASET_DIR / "data" / test_sample["pcd0"], weights_only=False)

In [40]:
file_path = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train") / f"{test_sample['scene_name']}.ply" 

In [41]:
ref_original = pcd = o3d.io.read_point_cloud(str(file_path))

In [42]:
test_ref.shape

(10788, 3)

In [43]:
np.asarray(ref_original.points).shape

(10788, 3)

In [44]:
np.mean(np.linalg.norm(np.asarray(ref_original.points) - test_ref, axis=1))

np.float64(0.03538939608742051)

In [45]:
src_pcd = o3d.geometry.PointCloud()
src_pcd.points = o3d.utility.Vector3dVector(test_src)
src_pcd.paint_uniform_color([1.0, 0.0, 0.0])
ref_pcd = o3d.geometry.PointCloud()
ref_pcd.points = o3d.utility.Vector3dVector(test_ref)
ref_pcd.paint_uniform_color([0.0, 1.0, 0.0])


PointCloud with 10788 points.

In [46]:
o3d.visualization.draw_plotly([ref_pcd, src_pcd])

In [47]:
t = test_sample["translation"]
rot = test_sample["rotation"]
T = np.eye(4, dtype=np.float64)
T[:3, :3] = rot
T[:3, 3] = t

In [48]:
# ref is mean face here so alignment won't look perfect
o3d.visualization.draw_plotly([ref_pcd, src_pcd.transform(T)])

In [49]:
# here we set ref to the full morphed sample of src to see perfect algignment

test_gt_morphed = torch.load(DATASET_DIR / "data" / test_sample["pcd_morphed"], weights_only=False)

morphed_pcd = o3d.geometry.PointCloud()
morphed_pcd.points = o3d.utility.Vector3dVector(test_gt_morphed)
morphed_pcd.paint_uniform_color([0.0, 1.0, 0.0]) 

src_pcd = o3d.geometry.PointCloud()
src_pcd.points = o3d.utility.Vector3dVector(test_src)
src_pcd.paint_uniform_color([1.0, 0.0, 0.0]) 

o3d.visualization.draw_plotly([morphed_pcd, src_pcd.transform(T)])

In [50]:
kept_indices = test_sample["kept_indices"]

In [51]:
r_errors = (np.hstack((test_src, np.ones((test_src.shape[0], 1)))) @ T.T)[:, :3] - test_ref[kept_indices]

In [52]:
np.linalg.norm(r_errors, axis=1).max()

np.float64(0.09510697991378737)

In [53]:
print(f"Reproyection error T_inv {np.linalg.norm((np.hstack((test_src, np.ones((test_src.shape[0], 1)))) @ T.T)[:, :3] - test_ref[kept_indices])}")


Reproyection error T_inv 1.8978435171052273


In [54]:
error = np.asarray(ref_pcd.points)[kept_indices] - np.asarray(src_pcd.points)

In [55]:
np.linalg.norm(error, axis=1).mean()

np.float64(0.02933863808159143)

In [56]:
_ex = train_metadata[np.random.randint(len(train_metadata))]
_src = torch.load(DATASET_DIR / "data" / _ex["pcd1"], weights_only=False)
_ref = torch.load(DATASET_DIR / "data" / _ex["pcd0"], weights_only=False)

_src_pcd = o3d.geometry.PointCloud()
_src_pcd.points = o3d.utility.Vector3dVector(_src)
_src_pcd.paint_uniform_color([1.0, 0.3, 0.3])  # red = source slice

_ref_pcd = o3d.geometry.PointCloud()
_ref_pcd.points = o3d.utility.Vector3dVector(_ref[_ex["kept_indices"]])
_ref_pcd.paint_uniform_color([0.3, 0.8, 0.3])  # green = ref points at slice locations

print(f"scene={_ex['scene_name']}  src_pts={len(_src)}  kept={len(_ex['kept_indices'])}")
o3d.visualization.draw_plotly([_src_pcd, _ref_pcd])


scene=2962  src_pts=3693  kept=3693


In [57]:
np.sum(np.linalg.norm(error, axis=1))

np.float64(95.70263742215124)